# Joins

The tracks table has IDs for artists and albums, but not their names. A
track record tells you `artist_id = 7`, which is not very helpful if you
are building a playlist display.

The data is deliberately split across tables. To put it back together, we
need joins.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
%sql sqlite:///../data/wavelength.sqlite

## Why tables are split

Imagine storing the artist name, country, and genre on every single track
row. If an artist has 50 tracks, you would have 50 copies of "The Midnight
Voltage", "rock", "UK". If the artist changes their name, you have to
update 50 rows and hope you do not miss one.

This is called **normalisation** -- splitting data into separate tables so
that each fact is stored exactly once. The trade-off is that you need joins
to reassemble the data when you query it. It is a good trade-off.

Let's remind ourselves of the relevant tables.

In [ ]:
%sqlcmd columns --table tracks

In [ ]:
%sqlcmd columns --table artists

The `tracks` table has `artist_id` and `album_id` columns that reference
the `artists` and `albums` tables. These are **foreign keys** -- they point
to the primary key in another table.

## INNER JOIN: combining matching rows

An INNER JOIN returns only rows where there is a match in both tables. Let's
get track titles alongside their artist names.

In [ ]:
%%sql
SELECT tracks.title, artists.name AS artist_name
FROM tracks
INNER JOIN artists ON tracks.artist_id = artists.artist_id
LIMIT 10;

The `ON` clause tells SQL how the tables relate: match rows where the
`artist_id` in `tracks` equals the `artist_id` in `artists`. Because both
tables have a column called `artist_id`, we have to be explicit:
`tracks.artist_id` and `artists.artist_id`.

## Table aliases

Writing `tracks.title` and `artists.name` every time gets tedious. We can
give tables short aliases, just like we alias columns. `t` for tracks, `a`
for artists -- short, clear, and much less typing.

In [ ]:
%%sql
SELECT t.title, a.name AS artist_name, t.genre, t.play_count
FROM tracks t
INNER JOIN artists a ON t.artist_id = a.artist_id
WHERE t.play_count IS NOT NULL
ORDER BY t.play_count DESC
LIMIT 10;

## LEFT JOIN: keeping all rows from the left table

An INNER JOIN only returns rows with matches in both tables. But what if
some artists have no albums? With an INNER JOIN, those artists would simply
disappear from the results.

A LEFT JOIN returns every row from the left table (the one in the FROM
clause), even if there is no matching row in the right table. Where there
is no match, the right-side columns come back as NULL.

In [ ]:
%%sql
SELECT a.name, a.genre, al.title AS album_title
FROM artists a
LEFT JOIN albums al ON a.artist_id = al.artist_id
WHERE al.album_id IS NULL;

These are artists in the database who have not released an album yet. With
an INNER JOIN, we would never see them. The LEFT JOIN preserves them, and
the `WHERE al.album_id IS NULL` filter isolates exactly the unmatched rows.

This is a very common pattern. "Find all X that have no Y" almost always
uses a LEFT JOIN followed by a NULL check.

Let's see the difference in counts directly.

In [ ]:
%%sql
SELECT
    (SELECT COUNT(DISTINCT a.artist_id)
     FROM artists a
     INNER JOIN albums al ON a.artist_id = al.artist_id) AS artists_with_albums,
    (SELECT COUNT(*) FROM artists) AS all_artists;

## Multi-table joins

You are not limited to two tables. Let's get track titles, artist names,
and album titles all in one query. Each JOIN adds one more table to the mix.

In [ ]:
%%sql
SELECT
    t.title AS track,
    a.name AS artist,
    al.title AS album,
    t.genre,
    t.play_count
FROM tracks t
INNER JOIN artists a ON t.artist_id = a.artist_id
INNER JOIN albums al ON t.album_id = al.album_id
WHERE t.play_count IS NOT NULL
ORDER BY t.play_count DESC
LIMIT 10;

You can chain as many joins as you need, though if you find yourself joining
seven or eight tables in one query, that is usually a sign to step back and
think about whether a simpler approach exists.

## The fan-out trap

This is a mistake that has caused wrong numbers in production dashboards at
real companies. It is subtle, and understanding it now will save you real
pain later.

Suppose we want to know the total duration of all tracks by each artist.
Let's first check a single track.

In [ ]:
%sql SELECT track_id, title, duration_seconds FROM tracks WHERE track_id = 1;

Now let's see how many times that track has been listened to.

In [ ]:
%sql SELECT COUNT(*) AS listen_count FROM listens WHERE track_id = 1;

If that track has been listened to, say, 20 times, then joining `tracks` to
`listens` will produce 20 rows for that single track. Now imagine you wrote
`SUM(t.duration_seconds)` -- you would get the track's duration multiplied
by the number of listens. That number is completely wrong, but it looks
plausible enough that it might slip into a report.

In [ ]:
%%sql
SELECT
    t.track_id,
    t.title,
    t.duration_seconds,
    COUNT(*) AS row_count
FROM tracks t
INNER JOIN listens l ON t.track_id = l.track_id
WHERE t.track_id = 1
GROUP BY t.track_id, t.title, t.duration_seconds;

This is the **fan-out trap**: when a one-to-many join multiplies your rows,
any aggregation on the "one" side produces inflated numbers.

The fix depends on what you actually want. If you want total duration of
tracks (each counted once), aggregate *before* joining. If you want total
listening time, sum the listens table's `duration_seconds` instead. We will
see proper aggregation techniques in the next notebook. For now, remember:
**every join can multiply your rows. Always check your row counts.**

## Self-joins

A self-join is when a table is joined to itself. This sounds strange, but
it is surprisingly useful. Let's find pairs of users who created playlists
containing the same track -- a signal that they have similar taste.

In [ ]:
%%sql
SELECT DISTINCT
    p1.user_id AS user_a,
    p2.user_id AS user_b,
    pt1.track_id
FROM playlist_tracks pt1
INNER JOIN playlist_tracks pt2
    ON pt1.track_id = pt2.track_id
    AND pt1.playlist_id != pt2.playlist_id
INNER JOIN playlists p1 ON pt1.playlist_id = p1.playlist_id
INNER JOIN playlists p2 ON pt2.playlist_id = p2.playlist_id
WHERE p1.user_id < p2.user_id
LIMIT 15;

The `playlist_tracks` table is joined to itself. We match rows that have
the same `track_id` but different `playlist_id` values, meaning the same
track appears in two different playlists. The `user_a < user_b` condition
avoids duplicates (Alice-Bob and Bob-Alice are the same pair).

Any time you need to compare rows within the same table, a self-join is
usually the answer.

## Exercises

### Exercise 1

Write a query that returns the track title and artist name for all tracks.
Use table aliases. Limit to 15 rows.

### Exercise 2

Find all albums released after 2020, showing the album title, release year,
and the artist name. Sort by release year descending.

### Exercise 3

Use a LEFT JOIN to find all artists who have no tracks. Show the artist
name and genre.

**Hint:** Join `artists` to `tracks` and look for NULLs on the tracks side.

### Exercise 4

Write a three-table join: show the username, track title, and the timestamp
of each listen. Join `listens` to `users` and `tracks`. Limit to 20 rows.

### Exercise 5

Find the top 10 most-played tracks, showing the track title, artist name,
album title, and play count. Use INNER JOINs for all three tables.

### Exercise 6

Find all tracks that appear in at least one playlist. Show the track title,
playlist name, and position. Join `tracks` to `playlist_tracks` to
`playlists`. Limit to 20 rows.

## Summary

- **INNER JOIN** returns only rows with matches in both tables
- **LEFT JOIN** returns all rows from the left table, with NULLs where the
  right table has no match
- **Table aliases** (`FROM tracks t`) make multi-table queries readable
- **Multi-table joins** chain multiple JOIN clauses to combine three or
  more tables
- **Self-joins** compare rows within the same table
- **The fan-out trap**: joins can multiply rows, inflating aggregations.
  Always check your row counts after a join

Next up: aggregation and grouping, where we start producing the summary
numbers that the analytics team actually needs.